# EVA — Дообучение в DataSphere
Не с нуля. Загружаем веса + снапшоты и продолжаем.


In [ ]:
# 1. Подготовка
import os
if not os.path.exists('/home/jupyter/EVA'):
    !git clone https://github.com/BlackCatSpb/FCF.git /home/jupyter/EVA
%cd /home/jupyter/EVA
!git pull 2>/dev/null; true
!pip install loguru tokenizers numpy faiss-cpu datasets huggingface_hub -q
!rm -rf /home/jupyter/EVA/snapshots
!mkdir -p /home/jupyter/EVA/snapshots /home/jupyter/EVA/logs

print("Готово")
!df -h /tmp

In [ ]:
# 2. Загрузка весов и снапшотов с локального ПК
import os, shutil

print("Жду загрузки файлов. Перетащите в JupyterLab:")
print("  1. weights.pt → /home/jupyter/EVA/checkpoints/lazy/final/")
print("  2. snapshots.pkl → /home/jupyter/EVA/checkpoints/lazy/final/")
print("  3. index.faiss → /home/jupyter/EVA/checkpoints/lazy/final/")
print("  4. meta.pkl → /home/jupyter/EVA/checkpoints/lazy/final/")
print("  5. config.json → /home/jupyter/EVA/checkpoints/lazy/final/")
print("  6. wiki_ru_large.txt → /home/jupyter/EVA/real_data/")

# Создаём структуру папок
for d in ['/home/jupyter/EVA/checkpoints/lazy/final', '/home/jupyter/EVA/real_data']:
    os.makedirs(d, exist_ok=True)

# Проверка после загрузки
weights = '/home/jupyter/EVA/checkpoints/lazy/final/weights.pt'
dataset = '/home/jupyter/EVA/real_data/wiki_ru_large.txt'

print(f"\nВеса: {'OK' if os.path.exists(weights) else 'НЕТ'} ({os.path.getsize(weights)//1024//1024 if os.path.exists(weights) else 0} MB)")
print(f"Датасет: {'OK' if os.path.exists(dataset) else 'НЕТ'} ({os.path.getsize(dataset)//1024//1024 if os.path.exists(dataset) else 0} MB)")

In [ ]:
# 3. Проверка GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "GPU не найден")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# 4. Загрузка модели и продолжение обучения
import sys
sys.path.insert(0, '/home/jupyter/EVA')

from eva.config import FCFConfig
from eva.primordial_layer import PrimordialLayer
from eva.tokenizer_utils import load_tokenizer
from eva.language_trainer import LanguageTrainer
from eva.unified_grammar import UnifiedStateGrammar
from eva.utils import load_primordial_layer
import torch, os, json, time, pickle, faiss, gc

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Загружаем модель целиком (веса + снапшоты)
checkpoint = '/home/jupyter/EVA/checkpoints/lazy/final'
layer = load_primordial_layer(checkpoint, PrimordialLayer)
if device == 'cuda':
    layer = layer.cuda()
print(f"Загружено: {layer.summary()}")
print(f"Снапшотов: {len(layer.state_storage.snapshots_meta)}")
print(f"Confidence: {layer.meta.average_confidence():.3f}")

tokenizer = load_tokenizer('/home/jupyter/EVA/tokenizer.json')
grammar = UnifiedStateGrammar(config=FCFConfig())

trainer = LanguageTrainer(
    layer=layer, tokenizer=tokenizer,
    checkpoint_dir='/home/jupyter/EVA/checkpoints',
    state_grammar=grammar, benchmark_interval=2000,
)
trainer.checkpoint_interval = 2000
trainer.gen_test_interval = 2000

# Дообучение: низкий LR
for pg in trainer.optimizer.param_groups:
    pg['lr'] = 1e-5

# Save function (веса в /tmp/, снапшоты на диск)
snapshot_dir = '/home/jupyter/EVA/snapshots'
os.makedirs(snapshot_dir, exist_ok=True)

def save(final=False):
    p = os.path.join(snapshot_dir, f"step_{trainer.step:06d}" if not final else "final")
    os.makedirs(p, exist_ok=True)
    with open(os.path.join(p, 'snapshots.pkl'), 'wb') as f:
        pickle.dump(trainer.layer.state_storage.snapshots_meta, f)
    if trainer.layer.state_storage.index is not None:
        faiss.write_index(trainer.layer.state_storage.index, os.path.join(p, 'index.faiss'))
    with open(os.path.join(p, 'meta.pkl'), 'wb') as f:
        pickle.dump({'usage_count': trainer.layer.meta.usage_count, 'confidence_history': trainer.layer.meta.confidence_history, 'created_at': trainer.layer.meta.created_at}, f)
    trainer.layer.config.to_json(os.path.join(p, 'config.json'))
    torch.save(trainer.layer.state_dict(), '/tmp/latest_weights.pt')
    s = {'step': trainer.step, 'snapshots': len(trainer.layer.state_storage.snapshots_meta), 'confidence': trainer.layer.meta.average_confidence(), 'timestamp': time.time()}
    with open(os.path.join(p, 'status.json'), 'w') as f:
        json.dump(s, f)
    print(f"[Save] step={trainer.step} snap={s['snapshots']} conf={s['confidence']:.3f}")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

trainer._save_checkpoint = save

train_file = '/home/jupyter/EVA/real_data/wiki_ru_large.txt'

print("\n" + "="*60)
print("  EVA — Дообучение (не с нуля)")
print("="*60)
print(f"  Устройство: {device}")
print(f"  LR: 1e-5")
print(f"  Веса: /tmp/latest_weights.pt")
print("="*60 + "\n")

stats = trainer.train(max_steps=50000, device=device, text_file=train_file, block_size=512, auto_stop=False)
print(f"\nОбучение завершено: {stats}")

In [ ]:
# 5. Тест генерации
prompts = [
    "История это наука которая изучает",
    "Математика помогает человечеству",
    "Природа Земли удивительна потому что",
    "Компьютеры обрабатывают данные с помощью",
    "Человек отличается от животных тем что",
]
layer.eval()
for prompt in prompts:
    enc = tokenizer.encode(prompt)
    ids = enc.ids if hasattr(enc, 'ids') else enc
    inp = torch.tensor([ids], dtype=torch.long)
    if device == 'cuda': inp = inp.cuda()
    out = layer.generate(inp, max_new_tokens=80, temperature=0.7, top_p=0.9)
    print(f"\nQ: {prompt}")
    print(f"A: {tokenizer.decode(out[0].tolist())}")

In [ ]:
# 6. Упаковка для скачивания
import os, shutil
dl = '/home/jupyter/eva_download'
if os.path.exists(dl): shutil.rmtree(dl)
os.makedirs(dl)

if os.path.exists('/tmp/latest_weights.pt'):
    shutil.copy('/tmp/latest_weights.pt', os.path.join(dl, 'weights.pt'))
    print(f"Веса: {os.path.getsize('/tmp/latest_weights.pt') // 1024 // 1024} MB")

steps = sorted([d for d in os.listdir(snapshot_dir) if d.startswith('step_')]) if os.path.exists(snapshot_dir) else []
if steps:
    shutil.copytree(os.path.join(snapshot_dir, steps[-1]), os.path.join(dl, 'snapshots'))
    print(f"Снапшоты из: {steps[-1]}")

!cd /home/jupyter && tar czf eva_model.tar.gz eva_download/
print(f"Готово: /home/jupyter/eva_model.tar.gz ({os.path.getsize('/home/jupyter/eva_model.tar.gz') // 1024 // 1024} MB)")
print("Скачайте: правый клик → Download")